# Policy RAG Assistant and Refund Agent 

## Implement(what we want to do in this Part_D)

## Refund rules the agent's tools must implement. 

Only completed orders can be refunded; a completed order delivered in more than 45 minutes is refunded 100% of its order_amount; delivered in 45 minutes or less → refund 0; a refund above INR 1000 is escalated to a human and not issued automatically. All simulated requests are treated as raised inside the policy's 7-day window.

In [54]:
"""mock_llm.py -- deterministic, offline stand-in for a hosted LLM chat API (no key, no network, no cost).
Interface mirrors a hosted chat API:  response = chat(messages, tools=None)
    response.choices[0].message.content / .tool_calls ; response.choices[0].finish_reason ; response.usage
Three behaviours are selected by a marker word in the FIRST (system) message:
    "CLASSIFY"  -> support-ticket classifier       "POLICY ASSISTANT" -> grounded policy answerer
    "AGENT"     -> refund agent that requests tools (ReAct style)"""
import json, math, re
from types import SimpleNamespace as NS

class TransientLLMError(Exception):
    """Simulated temporary failure (timeout / server busy). Retrying the same call succeeds."""

_state = {"calls": 0, "ok": 0}      # call counters that drive the simulated misbehaviour


def reset():
    """Restart the counters. Call once before each independent batch run."""
    _state["calls"] = 0; _state["ok"] = 0

CATS = ["billing", "delivery", "technical", "fraud"]
KW = {"fraud": {"unauthorized", "fraud", "hacked", "stolen"},
      "billing": {"charged", "invoice", "bill", "refund", "deducted", "payment"},
      "technical": {"app", "crash", "login", "otp", "error", "freezes"},
      "delivery": {"arrived", "rider", "delivery", "delivered", "late"}}

def est_tokens(text):
    return math.ceil(len((text or "").split()) * 1.3)

def _words(t):
    return set(re.findall(r"[a-z]+", (t or "").lower()))

def _resp(content=None, tool_calls=None, finish="stop", messages=()):
    prompt = sum(est_tokens(m.get("content") or json.dumps(m.get("tool_calls", ""), default=str)) for m in messages)
    comp = est_tokens(content or json.dumps(tool_calls or "", default=str))
    msg = NS(role="assistant", content=content, tool_calls=[NS(**tc) for tc in (tool_calls or [])])
    return NS(choices=[NS(message=msg, finish_reason=finish)],
              usage=NS(prompt_tokens=prompt, completion_tokens=comp, total_tokens=prompt + comp))

def _classify(messages):
    system, user = messages[0]["content"], messages[-1]["content"]
    _state["calls"] += 1
    if _state["calls"] % 9 == 0:                                 # every 9th call fails once
        raise TransientLLMError("503 server busy")
    _state["ok"] += 1
    order = ["fraud", "technical", "billing", "delivery"] if "EXAMPLES:" in system else ["billing", "fraud", "technical", "delivery"]
    w = _words(user)
    cat = next((c for c in order if KW[c] & w), "technical")
    if _state["ok"] % 13 == 0:                                   # chatty, non-JSON answer
        return _resp(f"The customer appears to have a {cat}-related problem.", messages=messages)
    if _state["ok"] % 17 == 0:                                   # JSON, but category outside the allowed list
        return _resp(json.dumps({"category": "shipping", "reason": "parcel movement"}), messages=messages)
    return _resp(json.dumps({"category": cat, "reason": "keyword match"}), messages=messages)

STOP = {"the", "a", "an", "is", "of", "to", "for", "in", "on", "how", "what", "when", "who", "are", "do", "does",
        "i", "my", "be", "can", "after", "from", "it", "or", "and", "at", "by", "which", "why", "if", "than", "that"}

def _stems(t):
    return {w[:5] for w in re.findall(r"[a-z]+", (t or "").lower()) if w not in STOP}

def _policy(messages):
    """Answers with the context sentence that best matches the question (rare words weigh more)."""
    user = messages[-1]["content"]
    m = re.search(r"CONTEXT:\s*(.*?)\s*QUESTION:\s*(.*)", user, re.S)
    context, question = (m.group(1), m.group(2)) if m else ("", user)
    sents = [s.strip() for s in re.split(r"(?<=[.])\s+", re.sub(r"\[[^\]]*\.txt\]", " ", context)) if s.strip()]
    qs = _stems(question)
    df = {q: sum(q in _stems(s) for s in sents) for q in qs}
    best, best_score = None, 0.0
    for s in sents:
        sc = sum(1.0 / df[q] for q in qs & _stems(s))
        if sc > best_score:
            best, best_score = s, sc
    if best is None:
        return _resp("I could not find that information in the policy documents.", messages=messages)
    return _resp(best, messages=messages)

def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    call = lambda n, a: _resp(None, [{"id": f"call_{len(tool_msgs)+1}", "name": n, "arguments": a}], "tool_calls", messages)
    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})
    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp("NEEDS_INFO: please share the order id.", messages=messages)
    last = tool_msgs[-1]; data = json.loads(last["content"]); name = last["name"]
    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)
        return call("search_policy", {"query": "refund policy late delivery"})
    order = json.loads(next(m for m in tool_msgs if m["name"] == "get_order")["content"])
    if name == "search_policy":
        if order["status"] != "completed":
            return _resp("DENIED: only completed orders can be refunded.", messages=messages)
        return call("calculate_refund", {"order_amount": order["order_amount"], "delivery_minutes": order["delivery_minutes"]})
    if name == "calculate_refund":
        amt = data["refund_inr"]
        if amt == 0:
            return _resp("DENIED: delivery was within 45 minutes.", messages=messages)
        if amt > 1000:
            return call("escalate_to_human", {"order_id": order["order_id"], "refund_inr": amt})
        return call("issue_refund", {"order_id": order["order_id"], "refund_inr": amt})
    if name == "issue_refund":
        return _resp(f"REFUND: INR {data['refund_inr']} issued.", messages=messages)
    if name == "escalate_to_human":
        return _resp("ESCALATE: refund above INR 1000 sent to a human lead.", messages=messages)
    return _resp("DENIED: unable to process.", messages=messages)

def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")


# 1. Chunk the documents

In [55]:
import pandas as pd

In [56]:
from pathlib import Path

In [57]:
# Set the main project directory
BASE_DIR = Path.cwd().parent

# Set the policy documents folder
POLICY_DIR = BASE_DIR / "data" / "policies"

print("Project Directory:", BASE_DIR)
print("Policy Directory:", POLICY_DIR)

Project Directory: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook
Policy Directory: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\data\policies


In [58]:
print("Current working directory:")
print(Path.cwd())

print("\nBase directory:")
print(BASE_DIR)

print("\nPolicy directory:")
print(POLICY_DIR)

print("\nDoes policy directory exist?")
print(POLICY_DIR.exists())

print("\nFiles and folders inside BASE_DIR:")
for item in BASE_DIR.iterdir():
    print(item.name)

Current working directory:
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code

Base directory:
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook

Policy directory:
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\data\policies

Does policy directory exist?
False

Files and folders inside BASE_DIR:
.ipynb_checkpoints
Chapters
Chapter_02 Python data Analysis
Cls_Exercise_Code
Code
data
Librarys .ipynb
masai
Masai_Code
Module
Module_3.ipynb


In [59]:
# Find all policy text files
policy_files = list(POLICY_DIR.glob("*.txt"))

# Store document contents
policy_documents = {}

for file_path in policy_files:

    content = file_path.read_text(encoding="utf-8")

    policy_documents[file_path.name] = content

In [60]:
# Display document word counts

for filename, content in policy_documents.items():

    word_count = len(content.split())

    print(f"{filename}: {word_count} words")

In [61]:
text = "Refunds are available for eligible orders"

words = text.split()

print(words)
print(len(words))

['Refunds', 'are', 'available', 'for', 'eligible', 'orders']
6


In [62]:
from pathlib import Path

# Search for the policies folder inside the project
BASE_DIR = Path.cwd()

while BASE_DIR.name != "Masai_Code" and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

POLICY_DIR = BASE_DIR / "data" / "policies"

print("Project folder:", BASE_DIR)
print("Policy folder:", POLICY_DIR)
print("Folder exists:", POLICY_DIR.exists())

print("\nPolicy files:")

if POLICY_DIR.exists():
    for file in POLICY_DIR.glob("*.txt"):
        print(file.name)

Project folder: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code
Policy folder: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\data\policies
Folder exists: True

Policy files:
delivery_policy.txt
escalation_policy.txt
payment_policy.txt
privacy_policy.txt
refund_policy.txt


In [63]:
# Find all policy text files
policy_files = list(POLICY_DIR.glob("*.txt"))

# Store document contents
policy_documents = {}

print("Total policy documents:", len(policy_files))

Total policy documents: 5


In [64]:
policy_files

[WindowsPath('c:/Users/PRAVEEN_KUMAR/jupyter_notebook/Jupyter notebook/Masai_Code/data/policies/delivery_policy.txt'),
 WindowsPath('c:/Users/PRAVEEN_KUMAR/jupyter_notebook/Jupyter notebook/Masai_Code/data/policies/escalation_policy.txt'),
 WindowsPath('c:/Users/PRAVEEN_KUMAR/jupyter_notebook/Jupyter notebook/Masai_Code/data/policies/payment_policy.txt'),
 WindowsPath('c:/Users/PRAVEEN_KUMAR/jupyter_notebook/Jupyter notebook/Masai_Code/data/policies/privacy_policy.txt'),
 WindowsPath('c:/Users/PRAVEEN_KUMAR/jupyter_notebook/Jupyter notebook/Masai_Code/data/policies/refund_policy.txt')]

In [65]:
print(BASE_DIR)
print(POLICY_DIR)
print(POLICY_DIR.exists())

c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\data\policies
True


In [66]:
# Find all policy text files
policy_files = list(POLICY_DIR.glob("*.txt"))

# Store document contents
policy_documents = {}

for file_path in policy_files:
    policy_documents[file_path.name] = file_path.read_text(
        encoding="utf-8"
    )

print("Total policy documents:", len(policy_documents))

for filename, content in policy_documents.items():
    print(filename, ":", len(content.split()), "words")

Total policy documents: 5
delivery_policy.txt : 100 words
escalation_policy.txt : 78 words
payment_policy.txt : 86 words
privacy_policy.txt : 68 words
refund_policy.txt : 119 words


In [67]:
# Define chunking function

def chunk_text(text, chunk_size=40, overlap=10):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = words[start:end]

        chunks.append(" ".join(chunk))

        # Stop after reaching the end of the document
        if end >= len(words):
            break

        # Move forward by chunk_size - overlap
        start += chunk_size - overlap

    return chunks

In [68]:
# Store all chunks with their source filenames

policy_chunks = []

for filename, content in policy_documents.items():

    chunks = chunk_text(content)

    for chunk in chunks:

        policy_chunks.append({
            "source": filename,
            "text": chunk
        })

print("Total chunks created:", len(policy_chunks))

Total chunks created: 15


In [69]:
# Count chunks from each policy document

chunk_counts = pd.Series(
    [chunk["source"] for chunk in policy_chunks]
).value_counts()

print(chunk_counts)

refund_policy.txt        4
delivery_policy.txt      3
escalation_policy.txt    3
payment_policy.txt       3
privacy_policy.txt       2
Name: count, dtype: int64


# 2. Vectorise and retrieve 

## 2.1 Library and purpose

TfidfVectorizer: Converts text documents into numerical vectors.

cosine_similarity: Measures how similar two text vectors are.

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [71]:
# Extract text from all policy chunks

chunk_texts = [
    chunk["text"]
    for chunk in policy_chunks
]

print("Total chunk texts:", len(chunk_texts))

print("\nFirst chunk:")
print(chunk_texts[0])

Total chunk texts: 15

First chunk:
Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,


In [72]:
# Create TF-IDF vectorizer

vectorizer = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF values
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("TF-IDF vectorization completed!")

print("Matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF vectorization completed!
Matrix shape: (15, 200)
Vocabulary size: 200


In [73]:
print("TF-IDF matrix:")
print(tfidf_matrix.toarray())

TF-IDF matrix:
[[0.         0.         0.15300471 ... 0.         0.1045224  0.        ]
 [0.         0.         0.31035858 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.11284736 0.         0.        ]
 ...
 [0.16350433 0.         0.         ... 0.         0.         0.        ]
 [0.29511408 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.29533159 0.24893717]]


## 2.2 Create a retrive funtion.

In [74]:
def retrieve(query, k=3):

    # Convert the query into a TF-IDF vector
    query_vector = vectorizer.transform([query])

    # Calculate similarity with all policy chunks
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    # Get indexes of the highest similarity scores
    top_indices = similarities.argsort()[::-1][:k]

    # Store the retrieved results
    results = []

    for index in top_indices:

        results.append({
            "source": policy_chunks[index]["source"],
            "score": float(similarities[index]),
            "text": policy_chunks[index]["text"]
        })

    return results

## 2.3 Test policy retrieval

**Conclusion**

    Check the query in 3 times and then all 3 times it will give the same responce it will be never change.
    That source is refund_policy.txt
    Similarity score is also closest score 0.271,0.251,0.2454

In [75]:
# Test policy retrieval

test_results = retrieve(
    "refund policy late delivery",
    k=3
)

for i, result in enumerate(test_results, start=1):

    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Similarity score:", result["score"])
    print("Text:", result["text"])


Result 1
Source: refund_policy.txt
Similarity score: 0.2719281507693355
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Result 2
Source: refund_policy.txt
Similarity score: 0.25137413448459545
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Result 3
Source: refund_policy.txt
Similarity score: 0.24540580878362586
Text: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original


# 3. Augment and answer

## 3.1 Create the system prompt

    Two Prompt
        1. POLICY_SYSTEM_PROMPT_1 it is a general prompt.

        2. POLICY_SYSTEM_PROMPT_2 is is the prompt imporve some tearms

        3. def answer(question) in this function we work with prompt 2.
            if we want to change the prompt in the funtion simply change the POLICY_SYSTEM_PROMPT_1 in the variable message.

In [76]:
POLICY_SYSTEM_PROMPT = """
You are a POLICY ASSISTANT.

Answer the user's question using only the supplied policy context.

Do not use outside knowledge or make up policy information.

Read the complete relevant policy text before answering.
When a policy states a specific condition and outcome,
include both clearly in your answer.

If the supplied context does not contain the answer,
respond exactly with:
I could not find the information in the provided policies.

Keep your answer clear, accurate, and concise.
"""

## 3.2 Create the answer function

In [77]:
def answer(question):

    # Retrieve the top 3 relevant policy chunks
    retrieved_chunks = retrieve(question, k=3)

    # Build the context with source filenames
    context = ""

    for chunk in retrieved_chunks:
        context += f"[{chunk['source']}]\n"
        context += chunk["text"] + "\n"

    # Construct the exact user message format
    user_message = (
        "CONTEXT:\n"
        + context
        + "\nQUESTION:\n"
        + question
    )

    # Send system and user messages to the mock LLM
    messages = [
        {
            "role": "system",
            "content": POLICY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Call the mock LLM
    response = chat(messages)

    # Extract the generated answer
    final_answer = response.choices[0].message.content

    # Return answer and source filenames
    sources = list(dict.fromkeys(
        chunk["source"] for chunk in retrieved_chunks
    ))

    return {
        "answer": final_answer,
        "sources": sources
    }

## 3.3 Test the RAG assistant

### Reason of the error.
### Required mock llm chat funtion.

    Method_1.
    
        1. So, it's through an error so import that first.
        2. Run that mock llm as a .py file and import os, get current directory.
        3. from pathlib import path
        4. from mock_llm import chat

    Method_2(Recommended)
    
        1. Run the entire mock.llm cell (It's Working)

In [78]:
print("chat" in globals())

True


In [79]:
print(chat)

<function chat at 0x000001FCFA520C10>


In [80]:
result = answer(
    "What happens if a completed order is delivered in more than 45 minutes?"
)

print("Answer:")
print(result["answer"])

print("\nSources used:")
print(result["sources"])

Answer:
If a completed order is delivered in more than 45 minutes, the customer
 
Delivery Policy.

Sources used:
['refund_policy.txt', 'delivery_policy.txt']


In [81]:
retrieved_chunks = retrieve(
    "What happens if a completed order is delivered in more than 45 minutes?",
    k=3
)

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", chunk["source"])
    print("Text:", chunk["text"])


--- Result 1 ---
Source: refund_policy.txt
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

--- Result 2 ---
Source: refund_policy.txt
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

--- Result 3 ---
Source: delivery_policy.txt
Text: Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,


In [82]:
result = answer(
    "What happens if a completed order is delivered in more than 45 minutes?"
)

print("Answer:")
print(result["answer"])

print("\nSources used:")
print(result["sources"])

Answer:
If a completed order is delivered in more than 45 minutes, the customer
 
Delivery Policy.

Sources used:
['refund_policy.txt', 'delivery_policy.txt']


# 4. Evaluate retrieval and grounding 

## 4.1 Load the evaluation dataset

In [83]:
# Define the evaluation file path
EVAL_FILE = BASE_DIR / "data" / "raw" / "policy_eval.csv"

# Load the evaluation dataset
policy_eval = pd.read_csv(EVAL_FILE)

# Display the dataset
display(policy_eval)

,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days


## 4.2 Check the dataset structure

In [84]:
print("Dataset shape:", policy_eval.shape)

print("\nColumn names:")
print(policy_eval.columns.tolist())

print("\nMissing values:")
print(policy_eval.isnull().sum())

Dataset shape: (14, 3)

Column names:
['question', 'expected_doc', 'expected_fact']

Missing values:
question         0
expected_doc     0
expected_fact    2
dtype: int64


## 4.3 Evaluate retrieval and answer grounding

In [85]:
# Store evaluation results
evaluation_results = []

# Test every question
for _, row in policy_eval.iterrows():

    question = row["question"]
    expected_doc = row["expected_doc"]
    expected_fact = row["expected_fact"]

    # Retrieve the top 3 policy chunks
    retrieved = retrieve(question, k=3)

    retrieved_sources = [
        chunk["source"] for chunk in retrieved
    ]

    # Generate the policy assistant's answer
    response = answer(question)
    generated_answer = response["answer"]

    # Check whether the expected document was retrieved
    if expected_doc != "NONE":
        retrieval_correct = expected_doc in retrieved_sources

        # Check whether the expected fact appears in the answer
        answer_correct = (
            expected_fact in generated_answer
        )

        fallback_correct = None

    else:
        retrieval_correct = None
        answer_correct = None

        # Check whether the assistant correctly refuses
        fallback_correct = generated_answer.startswith(
            "I could not find"
        )

    # Save the result
    evaluation_results.append({
        "question": question,
        "expected_doc": expected_doc,
        "retrieved_sources": retrieved_sources,
        "generated_answer": generated_answer,
        "retrieval_correct": retrieval_correct,
        "answer_correct": answer_correct,
        "fallback_correct": fallback_correct
    })

# Convert results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_results)

# Display evaluation results
display(evaluation_df)

,question,expected_doc,retrieved_sources,generated_answer,retrieval_correct,answer_correct,fallback_correct
0,What delivery time makes a completed order eli...,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, privacy...","order is delivered in more than 45 minutes, th...",True,True,None
1,Who must approve a refund above INR 1000?,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, refund_...",A refund above INR 1000 must be approved by a ...,True,True,None
2,How many days do I have to raise a refund requ...,refund_policy.txt,"[refund_policy.txt, privacy_policy.txt, refund...",A refund request must be raised within 7 days ...,True,True,None
3,What is the delivery fee on small orders?,delivery_policy.txt,"[delivery_policy.txt, delivery_policy.txt, pay...",A delivery fee of INR 25 is charged on orders ...,True,True,None
4,On orders of what value is delivery free?,delivery_policy.txt,"[delivery_policy.txt, delivery_policy.txt, pay...","of INR 25 is charged on orders below INR 199, ...",True,True,None
5,Why does the electronics delivery need a one t...,delivery_policy.txt,"[delivery_policy.txt, delivery_policy.txt, pay...",Electronics items are delivered only after the...,True,True,None
6,Is cash on delivery available for a large order?,payment_policy.txt,"[payment_policy.txt, payment_policy.txt, payme...",Cash on delivery is available only for orders ...,True,True,None
7,How soon is money returned when a payment fail...,payment_policy.txt,"[payment_policy.txt, payment_policy.txt, refun...","If a payment fails but money is deducted, the ...",True,True,None
8,Can wallet balance be refunded as cash?,payment_policy.txt,"[payment_policy.txt, payment_policy.txt, payme...",Wallet balance can be used for any order and i...,True,True,None
9,When are delivery addresses deleted from activ...,privacy_policy.txt,"[privacy_policy.txt, privacy_policy.txt, deliv...",Delivery addresses are deleted from active sys...,True,True,None


## 4.4 Calculate the evaluation summary

In [86]:
# Separate answerable and unanswerable questions
answerable = evaluation_df[
    evaluation_df["expected_doc"] != "NONE"
]

unanswerable = evaluation_df[
    evaluation_df["expected_doc"] == "NONE"
]

# Calculate evaluation scores
retrieval_score = answerable["retrieval_correct"].sum()

grounding_score = answerable["answer_correct"].sum()

fallback_score = unanswerable["fallback_correct"].sum()

# Display results
print("POLICY RAG EVALUATION SUMMARY")
print("--------------------------------")

print(
    f"Retrieval accuracy: {retrieval_score}/{len(answerable)}"
)

print(
    f"Answer grounding: {grounding_score}/{len(answerable)}"
)

print(
    f"Fallback accuracy: {fallback_score}/{len(unanswerable)}"
)

POLICY RAG EVALUATION SUMMARY
--------------------------------
Retrieval accuracy: 12/12
Answer grounding: 12/12
Fallback accuracy: 2/2


## 4.5 Create chunks with zero overlap

In [87]:
# Create a separate function for zero-overlap chunking
def chunk_text_no_overlap(text, chunk_size=40):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size):
        chunk = words[start:start + chunk_size]
        chunks.append(" ".join(chunk))

    return chunks

## 4.6 Build the zero-overlap chunks

In [88]:
# Store zero-overlap chunks separately
policy_chunks_no_overlap = []

for filename, content in policy_documents.items():

    chunks = chunk_text_no_overlap(content)

    for chunk in chunks:
        policy_chunks_no_overlap.append({
            "source": filename,
            "text": chunk
        })

print("Total zero-overlap chunks:", len(policy_chunks_no_overlap))

# Check chunks per document
display(
    pd.Series(
        [chunk["source"] for chunk in policy_chunks_no_overlap]
    ).value_counts()
)

Total zero-overlap chunks: 13


delivery_policy.txt      3
payment_policy.txt       3
refund_policy.txt        3
escalation_policy.txt    2
privacy_policy.txt       2
Name: count, dtype: int64

## 4.7 Create a separate TF-IDF vectorizer

In [89]:
# Extract zero-overlap chunk text
chunk_texts_no_overlap = [
    chunk["text"] for chunk in policy_chunks_no_overlap
]

# Create and fit a new TF-IDF vectorizer
vectorizer_no_overlap = TfidfVectorizer()

tfidf_matrix_no_overlap = vectorizer_no_overlap.fit_transform(
    chunk_texts_no_overlap
)

print("Zero-overlap vectorization completed!")

Zero-overlap vectorization completed!


## 4.8 Create a zero-overlap retrieval function

In [90]:
def retrieve_no_overlap(query, k=3):

    query_vector = vectorizer_no_overlap.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix_no_overlap
    )[0]

    top_indices = similarities.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": policy_chunks_no_overlap[index]["source"],
            "score": float(similarities[index]),
            "text": policy_chunks_no_overlap[index]["text"]
        })

    return results

## 4.9 Evaluate zero-overlap retrieval

In [91]:
zero_overlap_results = []

for _, row in policy_eval.iterrows():

    retrieved = retrieve_no_overlap(row["question"], k=3)

    retrieved_sources = [
        chunk["source"] for chunk in retrieved
    ]

    if row["expected_doc"] != "NONE":
        correct = row["expected_doc"] in retrieved_sources

        zero_overlap_results.append({
            "question": row["question"],
            "expected_doc": row["expected_doc"],
            "retrieved_sources": retrieved_sources,
            "retrieval_correct": correct
        })

zero_overlap_df = pd.DataFrame(zero_overlap_results)

display(zero_overlap_df)

print(
    "Zero-overlap retrieval:",
    zero_overlap_df["retrieval_correct"].sum(),
    "/",
    len(zero_overlap_df)
)

,question,expected_doc,retrieved_sources,retrieval_correct
0,What delivery time makes a completed order eli...,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, privacy...",True
1,Who must approve a refund above INR 1000?,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, refund_...",True
2,How many days do I have to raise a refund requ...,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, refund_...",True
3,What is the delivery fee on small orders?,delivery_policy.txt,"[delivery_policy.txt, payment_policy.txt, deli...",True
4,On orders of what value is delivery free?,delivery_policy.txt,"[delivery_policy.txt, delivery_policy.txt, pay...",True
5,Why does the electronics delivery need a one t...,delivery_policy.txt,"[delivery_policy.txt, delivery_policy.txt, del...",True
6,Is cash on delivery available for a large order?,payment_policy.txt,"[payment_policy.txt, payment_policy.txt, payme...",True
7,How soon is money returned when a payment fail...,payment_policy.txt,"[payment_policy.txt, refund_policy.txt, privac...",True
8,Can wallet balance be refunded as cash?,payment_policy.txt,"[payment_policy.txt, payment_policy.txt, payme...",True
9,When are delivery addresses deleted from activ...,privacy_policy.txt,"[privacy_policy.txt, delivery_policy.txt, esca...",True


Zero-overlap retrieval: 12 / 12


## 4.10 Compare both experiments

In [92]:
print("RETRIEVAL COMPARISON")
print("--------------------")

print(
    "With 10-word overlap:",
    answerable["retrieval_correct"].sum(),
    "/",
    len(answerable)
)

print(
    "With zero overlap:",
    zero_overlap_df["retrieval_correct"].sum(),
    "/",
    len(zero_overlap_df)
)

RETRIEVAL COMPARISON
--------------------
With 10-word overlap: 12 / 12
With zero overlap: 12 / 12


In [93]:
def answer_no_overlap(question):

    retrieved_chunks = retrieve_no_overlap(question, k=3)

    context = ""

    for chunk in retrieved_chunks:
        context += f"[{chunk['source']}]\n"
        context += chunk["text"] + "\n"

    user_message = (
        "CONTEXT:\n"
        + context
        + "\nQUESTION:\n"
        + question
    )

    messages = [
        {"role": "system", "content": POLICY_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    response = chat(messages)

    final_answer = response.choices[0].message.content

    sources = list(dict.fromkeys(
        chunk["source"] for chunk in retrieved_chunks
    ))

    return {
        "answer": final_answer,
        "sources": sources
    }

In [94]:
zero_overlap_answer_results = []

for _, row in policy_eval.iterrows():

    if row["expected_doc"] == "NONE":
        continue

    result = answer_no_overlap(row["question"])

    zero_overlap_answer_results.append({
        "question": row["question"],
        "expected_fact": row["expected_fact"],
        "generated_answer": result["answer"],
        "answer_correct": (
            row["expected_fact"] in result["answer"]
        )
    })

zero_overlap_answer_df = pd.DataFrame(
    zero_overlap_answer_results
)

display(zero_overlap_answer_df)

print(
    "Zero-overlap grounding:",
    zero_overlap_answer_df["answer_correct"].sum(),
    "/",
    len(zero_overlap_answer_df)
)

,question,expected_fact,generated_answer,answer_correct
0,What delivery time makes a completed order eli...,more than 45 minutes,receives a full refund of the order amount.,False
1,Who must approve a refund above INR 1000?,human support lead,A refund above INR\n \n1000 must be approved b...,True
2,How many days do I have to raise a refund requ...,within 7 days,A refund request must be raised within 7 days ...,True
3,What is the delivery fee on small orders?,INR 25,A delivery fee of INR 25 is charged on orders ...,True
4,On orders of what value is delivery free?,INR 199 or more,A delivery fee of INR 25 is charged on orders ...,True
5,Why does the electronics delivery need a one t...,one time password,Electronics items are delivered only after the...,True
6,Is cash on delivery available for a large order?,INR 2500,Cash on delivery is available only for orders ...,True
7,How soon is money returned when a payment fail...,48 hours,"If a payment fails but money is deducted, the ...",False
8,Can wallet balance be refunded as cash?,never refunded as cash,Wallet balance can be used for any order and i...,True
9,When are delivery addresses deleted from activ...,180 days,Delivery addresses are deleted from active sys...,True


Zero-overlap grounding: 10 / 12


## 4.11 Display failed question 

In [95]:
# Filter questions where zero-overlap answers failed
failed_questions = zero_overlap_answer_df[
    zero_overlap_answer_df["answer_correct"] == False
]

display(failed_questions)

,question,expected_fact,generated_answer,answer_correct
0,What delivery time makes a completed order eli...,more than 45 minutes,receives a full refund of the order amount.,False
7,How soon is money returned when a payment fail...,48 hours,"If a payment fails but money is deducted, the ...",False


# 5. Define tools and the tool map

## 5.1 Database connection

In [96]:
import sqlite3

# Define the database path
DB_PATH = BASE_DIR / "data" / "instamart.db"

# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Inspect the orders table
orders_schema = pd.read_sql_query(
    "PRAGMA table_info(orders)",
    conn
)

display(orders_schema)

# Close the connection
conn.close()

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_ts,TEXT,0,None,0
5,5,status,TEXT,0,None,0
6,6,discount_pct,INTEGER,0,None,0
7,7,order_amount,REAL,0,None,0
8,8,delivery_minutes,REAL,0,None,0
9,9,price_inr,INTEGER,0,None,0


## 5.2 Create a Tools

    1. get_order()
    2. search_policy()
    3. calculated_refund()
    4. issue_refund()
    5. escalate_to_human()

In [97]:
def get_order(order_id):

    # Connect to the database
    conn = sqlite3.connect(DB_PATH)

    # Fetch the required order details
    query = """
        SELECT order_id, status, order_amount, delivery_minutes
        FROM orders
        WHERE order_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(order_id,)
    )

    # Close database connection
    conn.close()

    # Check whether the order exists
    if result.empty:
        return {"error": "order not found"}

    # Convert the first matching row into a dictionary
    return result.iloc[0].to_dict()

In [98]:
# Load cleaned Part A order data
DATA_DIR = Path("data/clean")

orders = pd.read_csv(DATA_DIR / "orders.csv")

# Select 5 sample orders
sample_orders = orders.sample(n=5, random_state=42)

# Display sample orders
display(sample_orders)

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,price_inr
7244,ORD-102413,C0229,P024,1,2025-07-17 12:11:02,completed,20,72.0,25.0,90
8347,ORD-104654,C0438,P009,1,2025-08-02 20:56:48,completed,10,26.1,28.0,29
12173,ORD-103957,C0371,P007,1,2025-09-28 19:12:21,completed,15,102.0,22.0,120
9260,ORD-100960,C0089,P020,2,2025-08-16 09:17:58,completed,15,255.0,25.0,150
7791,ORD-106683,C0639,P024,2,2025-07-25 13:13:45,completed,25,135.0,15.0,90


In [99]:
# Select 5 sample orders from the cleaned order data
sample_orders = orders.sample(n=5, random_state=42)

# Display sample orders
display(sample_orders)

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,price_inr
7244,ORD-102413,C0229,P024,1,2025-07-17 12:11:02,completed,20,72.0,25.0,90
8347,ORD-104654,C0438,P009,1,2025-08-02 20:56:48,completed,10,26.1,28.0,29
12173,ORD-103957,C0371,P007,1,2025-09-28 19:12:21,completed,15,102.0,22.0,120
9260,ORD-100960,C0089,P020,2,2025-08-16 09:17:58,completed,15,255.0,25.0,150
7791,ORD-106683,C0639,P024,2,2025-07-25 13:13:45,completed,25,135.0,15.0,90


In [100]:
sample_orders

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,price_inr
7244,ORD-102413,C0229,P024,1,2025-07-17 12:11:02,completed,20,72.0,25.0,90
8347,ORD-104654,C0438,P009,1,2025-08-02 20:56:48,completed,10,26.1,28.0,29
12173,ORD-103957,C0371,P007,1,2025-09-28 19:12:21,completed,15,102.0,22.0,120
9260,ORD-100960,C0089,P020,2,2025-08-16 09:17:58,completed,15,255.0,25.0,150
7791,ORD-106683,C0639,P024,2,2025-07-25 13:13:45,completed,25,135.0,15.0,90


In [101]:
print("Available sample order IDs:")
print(sample_orders["order_id"].head(10).to_list())

print("\nDoes ORD001 exist?")
print(
    sample_orders["order_id"]
    .astype(str)
    .str.strip()
    .eq("ORD001")
    .any()
)

Available sample order IDs:
['ORD-102413', 'ORD-104654', 'ORD-103957', 'ORD-100960', 'ORD-106683']

Does ORD001 exist?
False


In [102]:
# Check an existing order
print(get_order("ORD-111396"))

# Check a non-existing order
print(get_order("INVALID_ORDER"))

{'order_id': 'ORD-111396', 'status': 'completed', 'order_amount': 1598.0, 'delivery_minutes': 25.0}
{'error': 'order not found'}


In [103]:
# Check an existing order
print(get_order("ORD001"))

# Check a non-existing order
print(get_order("INVALID_ORDER"))

{'error': 'order not found'}
{'error': 'order not found'}


In [104]:
conn = sqlite3.connect(DB_PATH)

sample_orders = pd.read_sql_query(
    """
    SELECT order_id, status, order_amount, delivery_minutes
    FROM orders
    LIMIT 5
    """,
    conn
)

conn.close()

display(sample_orders)

,order_id,status,order_amount,delivery_minutes
0,ORD-110093,completed,127.50,19.0
1,ORD-111396,completed,1598.00,25.0
2,ORD-102006,cancelled,21.75,NaN
3,ORD-105921,completed,86.40,33.0
4,ORD-108115,completed,104.40,21.0


In [105]:
print(get_order("ORD-110093"))

{'order_id': 'ORD-110093', 'status': 'completed', 'order_amount': 127.5, 'delivery_minutes': 19.0}


In [106]:
# Check the database path
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

# Connect to database
conn = sqlite3.connect(DB_PATH)

# Check total number of orders
order_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_orders FROM orders",
    conn
)

print("\nTotal orders:")
display(order_count)

# Display first 10 order IDs
sample_orders = pd.read_sql_query(
    "SELECT order_id, status, order_amount, delivery_minutes FROM orders LIMIT 10",
    conn
)

print("\nSample orders:")
display(sample_orders)

conn.close()

Database path: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\data\instamart.db
Database exists: True

Total orders:


,total_orders
0,12309



Sample orders:


,order_id,status,order_amount,delivery_minutes
0,ORD-110093,completed,127.50,19.0
1,ORD-111396,completed,1598.00,25.0
2,ORD-102006,cancelled,21.75,NaN
3,ORD-105921,completed,86.40,33.0
4,ORD-108115,completed,104.40,21.0
5,ORD-111469,completed,45.00,72.0
6,ORD-102351,cancelled,349.00,NaN
7,ORD-107027,completed,57.60,21.0
8,ORD-103459,completed,180.50,27.0
9,ORD-100900,completed,144.00,32.0


In [107]:
print(get_order("ORD-111396"))

{'order_id': 'ORD-111396', 'status': 'completed', 'order_amount': 1598.0, 'delivery_minutes': 25.0}


In [108]:
print(get_order("ORD-999999"))

{'error': 'order not found'}


In [109]:
def search_policy(query):

    # Retrieve the top 3 relevant policy chunks
    results = retrieve(query, k=3)

    # Return the results as a dictionary
    return {
        "results": results
    }

In [110]:
policy_result = search_policy(
    "What happens if a completed order is delivered in more than 45 minutes?"
)

print(policy_result)

{'results': [{'source': 'refund_policy.txt', 'score': 0.6918882501689527, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.476771879327522, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'delivery_policy.txt', 'score': 0.12366396486476816, 'text': 'Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,'}]}


## 5.2.1 Structured output for search_policy

In [111]:
policy_result = search_policy(
    "What happens if a completed order is delivered in more than 45 minutes?"
)

print("POLICY SEARCH RESULTS")
print("=" * 50)

for i, result in enumerate(policy_result["results"], start=1):

    print(f"\nResult {i}")
    print("-" * 50)

    print("Source:", result["source"])
    print("Similarity Score:", round(result["score"], 4))
    print("Policy Text:", result["text"])

POLICY SEARCH RESULTS

Result 1
--------------------------------------------------
Source: refund_policy.txt
Similarity Score: 0.6919
Policy Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Result 2
--------------------------------------------------
Source: refund_policy.txt
Similarity Score: 0.4768
Policy Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Result 3
--------------------------------------------------
Source: delivery_policy.txt
Similarity Score: 0.1237
Policy Text: Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mu

In [112]:
def calculate_refund(order_amount, delivery_minutes):

    # Convert inputs into numeric values
    order_amount = float(order_amount)
    delivery_minutes = float(delivery_minutes)

    # Check late delivery condition
    if delivery_minutes <= 45:
        return {
            "refund_inr": 0,
            "decision": "DENIED",
            "reason": "Delivery was completed within 45 minutes"
        }

    # Calculate full refund
    refund_inr = round(order_amount, 2)

    # Check whether human approval is required
    if refund_inr > 1000:
        return {
            "refund_inr": refund_inr,
            "decision": "ESCALATE",
            "reason": "Refund amount exceeds INR 1000"
        }

    # Automatic refund
    return {
        "refund_inr": refund_inr,
        "decision": "REFUND",
        "reason": "Completed order delivered after 45 minutes"
    }

In [113]:
print("TEST 1 — Automatic refund")
print(calculate_refund(128.25, 52))

print("\nTEST 2 — Human escalation")
print(calculate_refund(1999, 60))

print("\nTEST 3 — No refund")
print(calculate_refund(500, 30))

TEST 1 — Automatic refund
{'refund_inr': 128.25, 'decision': 'REFUND', 'reason': 'Completed order delivered after 45 minutes'}

TEST 2 — Human escalation
{'refund_inr': 1999.0, 'decision': 'ESCALATE', 'reason': 'Refund amount exceeds INR 1000'}

TEST 3 — No refund
{'refund_inr': 0, 'decision': 'DENIED', 'reason': 'Delivery was completed within 45 minutes'}


## 5.2.2 sturctured output for calculated_refund

In [114]:
# Store test cases
refund_tests = [
    {
        "test": "Automatic refund",
        "order_amount": 128.25,
        "delivery_minutes": 52
    },
    {
        "test": "Human escalation",
        "order_amount": 1999,
        "delivery_minutes": 60
    },
    {
        "test": "No refund",
        "order_amount": 500,
        "delivery_minutes": 30
    }
]

# Collect structured results
refund_results = []

for test in refund_tests:

    result = calculate_refund(
        test["order_amount"],
        test["delivery_minutes"]
    )

    refund_results.append({
        "Test Case": test["test"],
        "Order Amount (INR)": test["order_amount"],
        "Delivery Minutes": test["delivery_minutes"],
        "Refund Amount (INR)": result["refund_inr"],
        "Decision": result["decision"],
        "Reason": result["reason"]
    })

# Convert results into a DataFrame
refund_results_df = pd.DataFrame(refund_results)

# Display structured output
display(refund_results_df)

,Test Case,Order Amount (INR),Delivery Minutes,Refund Amount (INR),Decision,Reason
0,Automatic refund,128.25,52,128.25,REFUND,Completed order delivered after 45 minutes
1,Human escalation,1999.00,60,1999.00,ESCALATE,Refund amount exceeds INR 1000
2,No refund,500.00,30,0.00,DENIED,Delivery was completed within 45 minutes


In [115]:
import json
from datetime import datetime, timezone

# Define the output directory
OUTPUT_DIR = BASE_DIR / "outputs"

# Create the directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def issue_refund(order_id, refund_inr):

    # Prepare the refund record
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "tool": "issue_refund",
        "order_id": order_id,
        "refund_inr": round(float(refund_inr), 2)
    }

    # Define the ledger file path
    ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

    # Append the record to the file
    with ledger_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

    # Return the refund amount
    return {
        "refund_inr": round(float(refund_inr), 2)
    }

In [116]:
test_result = issue_refund("TEST-ORDER-001", 128.25)

print("Refund Tool Output:")
print(test_result)

Refund Tool Output:
{'refund_inr': 128.25}


In [117]:
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

with ledger_path.open("r", encoding="utf-8") as f:
    print(f.read())

{"timestamp": "2026-11-07T17:02:28.317162+00:00", "tool": "issue_refund", "order_id": "ORD-100048", "refund_inr": 128.25}
{"timestamp": "2026-11-07T17:02:28.317162+00:00", "tool": "issue_refund", "order_id": "ORD-100067", "refund_inr": 135.0}
{"timestamp": "2026-11-07T17:02:28.317162+00:00", "tool": "escalate_to_human", "order_id": "ORD-100145", "refund_inr": 1999.0}
{"timestamp": "2026-11-07T17:02:28.684469+00:00", "tool": "issue_refund", "order_id": "ORD-100048", "refund_inr": 128.25}
{"timestamp": "2026-11-08T06:54:52.090998+00:00", "tool": "issue_refund", "order_id": "TEST-ORDER-001", "refund_inr": 128.25}



In [118]:
def escalate_to_human(order_id, refund_inr):

    # Prepare the escalation record
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "tool": "escalate_to_human",
        "order_id": order_id,
        "refund_inr": round(float(refund_inr), 2)
    }

    # Define the ledger file path
    ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

    # Append the escalation record
    with ledger_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

    # Return the refund amount
    return {
        "refund_inr": round(float(refund_inr), 2)
    }

In [119]:
escalation_result = escalate_to_human("TEST-ORDER-002", 1999.00)

print("Escalation Tool Output:")
print(escalation_result)

Escalation Tool Output:
{'refund_inr': 1999.0}


In [120]:
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

ledger_records = []

with ledger_path.open("r", encoding="utf-8") as f:
    for line in f:
        ledger_records.append(json.loads(line))

ledger_df = pd.DataFrame(ledger_records)

display(ledger_df)

,timestamp,tool,order_id,refund_inr
0,2026-11-07T17:02:28.317162+00:00,issue_refund,ORD-100048,128.25
1,2026-11-07T17:02:28.317162+00:00,issue_refund,ORD-100067,135.00
2,2026-11-07T17:02:28.317162+00:00,escalate_to_human,ORD-100145,1999.00
3,2026-11-07T17:02:28.684469+00:00,issue_refund,ORD-100048,128.25
4,2026-11-08T06:54:52.090998+00:00,issue_refund,TEST-ORDER-001,128.25
5,2026-11-08T06:54:52.156971+00:00,escalate_to_human,TEST-ORDER-002,1999.00


## 5.3 Connect all five tools

In [121]:
TOOL_MAP = {
    "get_order": get_order,
    "search_policy": search_policy,
    "calculate_refund": calculate_refund,
    "issue_refund": issue_refund,
    "escalate_to_human": escalate_to_human
}

print("Available tools:")
for tool_name in TOOL_MAP:
    print(tool_name)

Available tools:
get_order
search_policy
calculate_refund
issue_refund
escalate_to_human


In [122]:
TOOL_MAP["get_order"]

<function __main__.get_order(order_id)>

In [123]:
TOOLS_SPEC = [

    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Retrieve order details using the order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "Unique order ID"
                    }
                },
                "required": ["order_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "search_policy",
            "description": "Search the policy documents for relevant information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Question or search query about a policy"
                    }
                },
                "required": ["query"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "calculate_refund",
            "description": "Calculate the refund amount and decide whether to refund, deny, or escalate.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_amount": {
                        "type": "number",
                        "description": "Total order amount in INR"
                    },
                    "delivery_minutes": {
                        "type": "number",
                        "description": "Delivery time in minutes"
                    }
                },
                "required": ["order_amount", "delivery_minutes"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "issue_refund",
            "description": "Record an approved automatic refund in the refund ledger.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "Order ID"
                    },
                    "refund_inr": {
                        "type": "number",
                        "description": "Approved refund amount in INR"
                    }
                },
                "required": ["order_id", "refund_inr"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Record a refund request that requires human review.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "Order ID"
                    },
                    "refund_inr": {
                        "type": "number",
                        "description": "Refund amount requiring human review"
                    }
                },
                "required": ["order_id", "refund_inr"]
            }
        }
    }

]

## 5.4 Description of the Tools

In [124]:
print("TOTAL TOOLS:", len(TOOLS_SPEC))

for tool in TOOLS_SPEC:
    function_info = tool["function"]

    print("\nTool Name:", function_info["name"])
    print("Description:", function_info["description"])
    print("Required Parameters:", function_info["parameters"]["required"])

TOTAL TOOLS: 5

Tool Name: get_order
Description: Retrieve order details using the order ID.
Required Parameters: ['order_id']

Tool Name: search_policy
Description: Search the policy documents for relevant information.
Required Parameters: ['query']

Tool Name: calculate_refund
Description: Calculate the refund amount and decide whether to refund, deny, or escalate.
Required Parameters: ['order_amount', 'delivery_minutes']

Tool Name: issue_refund
Description: Record an approved automatic refund in the refund ledger.
Required Parameters: ['order_id', 'refund_inr']

Tool Name: escalate_to_human
Description: Record a refund request that requires human review.
Required Parameters: ['order_id', 'refund_inr']


# 6. Run the ReAct loop

In [125]:
reset()

test_messages = [
    {
        "role": "system",
        "content": """
        AGENT

        You are a refund processing agent.
        Check the order details and follow the refund policy.
        Use the available tools to process the request.
        """
    },
    {
        "role": "user",
        "content": "Check the order ORD-111396 and tell me whether it qualifies for a refund."
    }
]

test_response = chat(test_messages, tools=TOOLS_SPEC)

print("Response object:")
print(test_response)

print("\nToken usage:")
print(test_response.usage.total_tokens)

print("\nAssistant message:")
print(test_response.choices[0].message)

Response object:
namespace(choices=[namespace(message=namespace(role='assistant', content=None, tool_calls=[namespace(id='call_1', name='get_order', arguments={'order_id': 'ORD-111396'})]), finish_reason='tool_calls')], usage=namespace(prompt_tokens=49, completion_tokens=10, total_tokens=59))

Token usage:
59

Assistant message:
namespace(role='assistant', content=None, tool_calls=[namespace(id='call_1', name='get_order', arguments={'order_id': 'ORD-111396'})])


In [126]:
tool_name = "get_order"
arguments = {"order_id": "ORD-111396"}

tool_function = TOOL_MAP[tool_name]

tool_result = tool_function(**arguments)

print(tool_result)

{'order_id': 'ORD-111396', 'status': 'completed', 'order_amount': 1598.0, 'delivery_minutes': 25.0}


In [127]:
get_order(order_id="ORD-111396")

{'order_id': 'ORD-111396',
 'status': 'completed',
 'order_amount': 1598.0,
 'delivery_minutes': 25.0}

In [128]:
tool_name = test_response.choices[0].message.tool_calls[0].name

arguments = test_response.choices[0].message.tool_calls[0].arguments

tool_function = TOOL_MAP[tool_name]

tool_result = tool_function(**arguments)

print("Tool Name:", tool_name)
print("Arguments:", arguments)
print("Tool Result:", tool_result)

Tool Name: get_order
Arguments: {'order_id': 'ORD-111396'}
Tool Result: {'order_id': 'ORD-111396', 'status': 'completed', 'order_amount': 1598.0, 'delivery_minutes': 25.0}


## 6.1 Before go to the Tools part check the llm part.

## Lets inspect how llm work

1. Read the requested tool name.
2. Extract its arguments.
3. Execute the Python function.
4. Obtain the tool result.

In [129]:
import inspect

print(inspect.getsource(_agent))

def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    call = lambda n, a: _resp(None, [{"id": f"call_{len(tool_msgs)+1}", "name": n, "arguments": a}], "tool_calls", messages)
    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})
    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp("NEEDS_INFO: please share the order id.", messages=messages)
    last = tool_msgs[-1]; data = json.loads(last["content"]); name = last["name"]
    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)
        return call("search_policy", {"query": "refund policy late delivery"})
    order = json.loads(next(m for m in tool_msgs if m["name"] == "get_order")["content"])
    if name == "search_policy":
        if o

## 6.2 Build the run agent

In [130]:
def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    # 1. Start the conversation
    messages = [
        {
            "role": "system",
            "content": """
            AGENT

            You are an Instamart refund processing agent.
            Follow the refund policy and use the available tools.
            Do not invent order information or refund decisions.
            """
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # 2. Initialize tracking variables
    steps = []
    llm_calls = 0
    tokens_used = 0
    final_answer = None

    # 3. ReAct loop
    for iteration in range(max_iterations):

        # Call the mock LLM
        response = chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        # Get the assistant message
        assistant_message = response.choices[0].message

        # 4. Check whether the LLM returned a final answer
        if not assistant_message.tool_calls:

            final_answer = assistant_message.content
            break

        # 5. Store the assistant tool-call message
        tool_calls_data = []

        for tool_call in assistant_message.tool_calls:
            tool_calls_data.append({
                "id": tool_call.id,
                "name": tool_call.name,
                "arguments": tool_call.arguments
            })

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": tool_calls_data
        })

        # 6. Execute each requested tool
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.name
            arguments = tool_call.arguments

            # Find the corresponding Python function
            tool_function = TOOL_MAP[tool_name]

            # Execute the function
            observation = tool_function(**arguments)

            # Record the execution
            steps.append({
                "step": iteration + 1,
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            # 7. Send the tool result back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

    # 8. Handle the case where no final answer was produced
    if final_answer is None:
        status = "FALLBACK_MAX_ITERATIONS"
        final_answer = (
            "FALLBACK_MAX_ITERATIONS: "
            "Request requires human review."
        )
    else:
        # Extract status from the final answer
        status = final_answer.split(":", 1)[0].strip()

    # 9. Return the complete agent result
    return {
        "scenario_id": scenario_id,
        "status": status,
        "final_answer": final_answer,
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used
    }

In [131]:
print(run_agent)

<function run_agent at 0x000001FCAAD8B6D0>


## 6.3 Take the agent scenarios.csv from raw data then use this for our RAG Application. 

In [132]:
SCENARIO_FILE = BASE_DIR / "data" / "raw" / "agent_scenarios.csv"

agent_scenarios = pd.read_csv(SCENARIO_FILE)

display(agent_scenarios)

,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [133]:
first_scenario = agent_scenarios.iloc[0]

display(first_scenario.to_frame(name="Value"))

,Value
scenario_id,S1
customer_id,C0007
message,My order ORD-100048 reached me very late. I wo...
expected_outcome,REFUND
expected_refund_inr,128.25


In [134]:
first_scenario = agent_scenarios.iloc[5]

display(first_scenario.to_frame(name="Value"))

,Value
scenario_id,S6
customer_id,C0001
message,Refund for order ORD-999999 please.
expected_outcome,NOT_FOUND
expected_refund_inr,0.0


In [135]:
first_scenario = agent_scenarios.iloc[6]

display(first_scenario.to_frame(name="Value"))

,Value
scenario_id,S7
customer_id,C0007
message,[LOOP] Refund order ORD-100048.
expected_outcome,FALLBACK
expected_refund_inr,0.0


In [136]:
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

ledger_path.write_text("", encoding="utf-8")

print("Test ledger cleared successfully!")

Test ledger cleared successfully!


## 6.4 Clear the test ledger

In [137]:
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

ledger_path.write_text("", encoding="utf-8")

print("Test ledger cleared successfully!")

Test ledger cleared successfully!


In [138]:
s1_result = run_agent(
    user_message=first_scenario["message"],
    scenario_id=first_scenario["scenario_id"]
)

print("SCENARIO ID:", s1_result["scenario_id"])
print("STATUS:", s1_result["status"])
print("FINAL ANSWER:", s1_result["final_answer"])
print("LLM CALLS:", s1_result["llm_calls"])
print("TOKENS USED:", s1_result["tokens_used"])

SCENARIO ID: S7
STATUS: FALLBACK_MAX_ITERATIONS
FINAL ANSWER: FALLBACK_MAX_ITERATIONS: Request requires human review.
LLM CALLS: 6
TOKENS USED: 3120


In [139]:
s1_steps_df = pd.DataFrame(s1_result["steps"])

display(s1_steps_df)

,step,tool,arguments,observation
0,1,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."
1,2,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."
2,3,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."
3,4,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."
4,5,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."
5,6,search_policy,{'query': 'refund policy'},"{'results': [{'source': 'refund_policy.txt', '..."


In [140]:
def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")

    tool_msgs = [m for m in messages if m["role"] == "tool"]

    call = lambda n, a: _resp(
        None,
        [{
            "id": f"call_{len(tool_msgs)+1}",
            "name": n,
            "arguments": a
        }],
        "tool_calls",
        messages
    )

    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})

    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp(
            "NEEDS_INFO: please share the order id.",
            messages=messages
        )

    last = tool_msgs[-1]
    data = json.loads(last["content"])
    name = last["name"]

    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)

        return call("search_policy", {"query": "refund policy late delivery"})

    order = json.loads(
        next(m for m in tool_msgs if m["name"] == "get_order")["content"]
    )

    if name == "search_policy":
        if order["status"] != "completed":
            return _resp(
                "DENIED: only completed orders can be refunded.",
                messages=messages
            )

        return call(
            "calculate_refund",
            {
                "order_amount": order["order_amount"],
                "delivery_minutes": order["delivery_minutes"]
            }
        )

    if name == "calculate_refund":
        amt = data["refund_inr"]

        if amt == 0:
            return _resp(
                "DENIED: delivery was within 45 minutes.",
                messages=messages
            )

        if amt > 1000:
            return call(
                "escalate_to_human",
                {
                    "order_id": order["order_id"],
                    "refund_inr": amt
                }
            )

        return call(
            "issue_refund",
            {
                "order_id": order["order_id"],
                "refund_inr": amt
            }
        )

    if name == "issue_refund":
        return _resp(
            f"REFUND: INR {data['refund_inr']} issued.",
            messages=messages
        )

    if name == "escalate_to_human":
        return _resp(
            "ESCALATE: refund above INR 1000 sent to a human lead.",
            messages=messages
        )

    return _resp("DENIED: unable to process.", messages=messages)

In [141]:
# Select S1 explicitly
s1_row = agent_scenarios[
    agent_scenarios["scenario_id"] == "S1"
].iloc[0]

print("Scenario:", s1_row["scenario_id"])
print("Customer message:", s1_row["message"])
print("Expected outcome:", s1_row["expected_outcome"])
print("Expected refund:", s1_row["expected_refund_inr"])

Scenario: S1
Customer message: My order ORD-100048 reached me very late. I would like a refund.
Expected outcome: REFUND
Expected refund: 128.25


In [142]:
s1_result = run_agent(
    user_message=s1_row["message"],
    scenario_id=s1_row["scenario_id"]
)

print("STATUS:", s1_result["status"])
print("FINAL ANSWER:", s1_result["final_answer"])
print("LLM CALLS:", s1_result["llm_calls"])
print("TOKENS USED:", s1_result["tokens_used"])

STATUS: REFUND
FINAL ANSWER: REFUND: INR 128.25 issued.
LLM CALLS: 5
TOKENS USED: 1021


In [143]:
s1_steps_df = pd.DataFrame(s1_result["steps"])

display(s1_steps_df)

,step,tool,arguments,observation
0,1,get_order,{'order_id': 'ORD-100048'},"{'order_id': 'ORD-100048', 'status': 'complete..."
1,2,search_policy,{'query': 'refund policy late delivery'},"{'results': [{'source': 'refund_policy.txt', '..."
2,3,calculate_refund,"{'order_amount': 128.25, 'delivery_minutes': 6...","{'refund_inr': 128.25, 'decision': 'REFUND', '..."
3,4,issue_refund,"{'order_id': 'ORD-100048', 'refund_inr': 128.25}",{'refund_inr': 128.25}


## 6.5 Run all seven scenarios

In [144]:
# Clear previous test records
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"
ledger_path.write_text("", encoding="utf-8")

# Prepare a fresh agent run log
agent_runs_path = OUTPUT_DIR / "agent_runs.jsonl"
agent_runs_path.write_text("", encoding="utf-8")

print("Output files prepared successfully!")

Output files prepared successfully!


In [145]:
agent_results = []

for _, row in agent_scenarios.iterrows():

    print("Running scenario:", row["scenario_id"])

    # Execute the agent
    result = run_agent(
        user_message=row["message"],
        scenario_id=row["scenario_id"]
    )

    agent_results.append(result)

    # Append one JSON record per scenario
    with agent_runs_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(result) + "\n")

print("\nAll scenarios completed!")

Running scenario: S1
Running scenario: S2
Running scenario: S3
Running scenario: S4
Running scenario: S5
Running scenario: S6
Running scenario: S7

All scenarios completed!


In [146]:
scenario_summary = []

for result in agent_results:

    refund_amount = None

    for step in result["steps"]:
        if step["tool"] in [
            "calculate_refund",
            "issue_refund",
            "escalate_to_human"
        ]:
            refund_amount = step["observation"].get("refund_inr")

    scenario_summary.append({
        "Scenario ID": result["scenario_id"],
        "Status": result["status"],
        "Refund Amount (INR)": refund_amount,
        "LLM Calls": result["llm_calls"],
        "Tokens Used": result["tokens_used"]
    })

scenario_summary_df = pd.DataFrame(scenario_summary)

display(scenario_summary_df)

,Scenario ID,Status,Refund Amount (INR),LLM Calls,Tokens Used
0,S1,REFUND,128.25,5,1021
1,S2,REFUND,135.00,5,1021
2,S3,ESCALATE,1999.00,5,1004
3,S4,DENIED,0.00,4,697
4,S5,DENIED,NaN,3,397
5,S6,NOT_FOUND,NaN,2,112
6,S7,FALLBACK_MAX_ITERATIONS,NaN,6,3120


In [147]:
comparison_df = agent_scenarios[
    ["scenario_id", "expected_outcome", "expected_refund_inr"]
].merge(
    scenario_summary_df,
    left_on="scenario_id",
    right_on="Scenario ID"
)

display(comparison_df)

,scenario_id,expected_outcome,expected_refund_inr,Scenario ID,Status,Refund Amount (INR),LLM Calls,Tokens Used
0,S1,REFUND,128.25,S1,REFUND,128.25,5,1021
1,S2,REFUND,135.00,S2,REFUND,135.00,5,1021
2,S3,ESCALATE,1999.00,S3,ESCALATE,1999.00,5,1004
3,S4,DENIED,0.00,S4,DENIED,0.00,4,697
4,S5,DENIED,0.00,S5,DENIED,NaN,3,397
5,S6,NOT_FOUND,0.00,S6,NOT_FOUND,NaN,2,112
6,S7,FALLBACK,0.00,S7,FALLBACK_MAX_ITERATIONS,NaN,6,3120


## ------ReAct agent is working.------

# 7. Stop safely 

## SUMMARY
    (a) Maximum iterations	FALLBACK_MAX_ITERATIONS
    (b) Token limit	FALLBACK_TOKEN_LIMIT
    (c) Time limit	FALLBACK_TIMEOUT

# 7.1 Update run_agent() with safety limits

In [148]:
import time

def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    # 1. Start execution timer
    start_time = time.monotonic()

    # 2. Initialize conversation
    messages = [
        {
            "role": "system",
            "content": """
            AGENT

            You are an Instamart refund processing agent.
            Follow the refund policy and use available tools.
            Do not invent order information or refund decisions.
            """
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # 3. Initialize tracking variables
    steps = []
    llm_calls = 0
    tokens_used = 0
    final_answer = None
    status = None

    # 4. ReAct loop
    for iteration in range(max_iterations):

        # Check time limit before making a model call
        elapsed_time = time.monotonic() - start_time

        if elapsed_time >= time_limit_seconds:
            status = "FALLBACK_TIMEOUT"
            final_answer = (
                "FALLBACK_TIMEOUT: "
                "Request requires human review."
            )
            break

        # Call the mock LLM
        response = chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        # Check token limit before executing any requested tool
        if tokens_used > token_limit:
            status = "FALLBACK_TOKEN_LIMIT"
            final_answer = (
                "FALLBACK_TOKEN_LIMIT: "
                "Request requires human review."
            )
            break

        # Get assistant message
        assistant_message = response.choices[0].message

        # Check for final answer
        if not assistant_message.tool_calls:
            final_answer = assistant_message.content
            break

        # Store assistant tool-call message
        tool_calls_data = []

        for tool_call in assistant_message.tool_calls:
            tool_calls_data.append({
                "id": tool_call.id,
                "name": tool_call.name,
                "arguments": tool_call.arguments
            })

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": tool_calls_data
        })

        # Execute requested tools
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.name
            arguments = tool_call.arguments

            tool_function = TOOL_MAP[tool_name]

            observation = tool_function(**arguments)

            # Record tool execution
            steps.append({
                "step": len(steps) + 1,
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            # Send observation back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

    # 5. Handle maximum iteration fallback
    if final_answer is None:
        if status is None:
            status = "FALLBACK_MAX_ITERATIONS"
            final_answer = (
                "FALLBACK_MAX_ITERATIONS: "
                "Request requires human review."
            )

    # 6. Extract normal final status
    if status is None:
        status = final_answer.split(":", 1)[0].strip()

    # 7. Return complete result
    return {
        "scenario_id": scenario_id,
        "status": status,
        "final_answer": final_answer,
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used
    }

In [149]:
print(run_agent)

<function run_agent at 0x000001FCFA521000>


## 7.2 FALLBACK_MAX_ITERATIONS

In [150]:
s7_row = agent_scenarios[
    agent_scenarios["scenario_id"] == "S7"
].iloc[0]

s7_safe_result = run_agent(
    user_message=s7_row["message"],
    scenario_id="S7"
)

print("Status:", s7_safe_result["status"])
print("Final answer:", s7_safe_result["final_answer"])
print("LLM calls:", s7_safe_result["llm_calls"])
print("Tokens used:", s7_safe_result["tokens_used"])
print("Tools executed:", len(s7_safe_result["steps"]))

Status: FALLBACK_MAX_ITERATIONS
Final answer: FALLBACK_MAX_ITERATIONS: Request requires human review.
LLM calls: 6
Tokens used: 3114
Tools executed: 6


## 7.3 FALLBACK_TOKEN_LIMIT

In [151]:
s1_row = agent_scenarios[
    agent_scenarios["scenario_id"] == "S1"
].iloc[0]

s1_token_result = run_agent(
    user_message=s1_row["message"],
    scenario_id="S1",
    token_limit=200
)

print("Status:", s1_token_result["status"])
print("Final answer:", s1_token_result["final_answer"])
print("LLM calls:", s1_token_result["llm_calls"])
print("Tokens used:", s1_token_result["tokens_used"])
print("Tools executed:", len(s1_token_result["steps"]))

display(pd.DataFrame(s1_token_result["steps"]))

Status: FALLBACK_TOKEN_LIMIT
Final answer: FALLBACK_TOKEN_LIMIT: Request requires human review.
LLM calls: 3
Tokens used: 411
Tools executed: 2


,step,tool,arguments,observation
0,1,get_order,{'order_id': 'ORD-100048'},"{'order_id': 'ORD-100048', 'status': 'complete..."
1,2,search_policy,{'query': 'refund policy late delivery'},"{'results': [{'source': 'refund_policy.txt', '..."


## 7.4 Test the timeout condition

In [152]:
s1_timeout_result = run_agent(
    user_message=s1_row["message"],
    scenario_id="S1",
    time_limit_seconds=0
)

print("Status:", s1_timeout_result["status"])
print("Final answer:", s1_timeout_result["final_answer"])
print("LLM calls:", s1_timeout_result["llm_calls"])
print("Tokens used:", s1_timeout_result["tokens_used"])
print("Tools executed:", len(s1_timeout_result["steps"]))

Status: FALLBACK_TIMEOUT
Final answer: FALLBACK_TIMEOUT: Request requires human review.
LLM calls: 0
Tokens used: 0
Tools executed: 0


# 8. Make the run resumable 


## 8.1 Create the state folder and def save_checkpoint funtion

In [153]:
# Create the state folder
STATE_DIR = BASE_DIR / "state"

STATE_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint file location
CHECKPOINT_DIR = STATE_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("State folder:", STATE_DIR)
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Checkpoint folder exists:", CHECKPOINT_DIR.exists())

State folder: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\state
Checkpoint folder: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\state\checkpoints
Checkpoint folder exists: True


In [154]:
import json
from datetime import datetime, timezone

def save_checkpoint(
    run_id,
    scenario_id,
    messages,
    tokens_used,
    steps
):
    checkpoint = {
        "run_id": run_id,
        "scenario_id": scenario_id,
        "messages": messages,
        "tokens_used": tokens_used,
        "step_number": len(steps),
        "steps": steps,
        "saved_at": datetime.now(timezone.utc).isoformat()
    }

    checkpoint_path = CHECKPOINT_DIR / f"{run_id}.json"

    with checkpoint_path.open("w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=4)

    return checkpoint_path

In [155]:
print(save_checkpoint)

<function save_checkpoint at 0x000001FCFA520DC0>


## 8.2 Create the checkpoint-loading function
    We have created a function to save the agent's progress. Now we need another function to read that saved progress when the agent restarts.

In [156]:
def load_checkpoint(run_id):

    checkpoint_path = CHECKPOINT_DIR / f"{run_id}.json"

    # Check whether the checkpoint exists
    if not checkpoint_path.exists():
        return {"error": "checkpoint not found"}

    # Read the saved checkpoint
    with checkpoint_path.open("r", encoding="utf-8") as f:
        checkpoint = json.load(f)

    return checkpoint

In [157]:
print(load_checkpoint)

<function load_checkpoint at 0x000001FCAAB90F70>


## 8.3 Create a function to execute one tool

In [158]:
def execute_tool_call(tool_call, messages, steps):

    tool_name = tool_call.name
    arguments = tool_call.arguments

    # Find the correct Python function
    tool_function = TOOL_MAP[tool_name]

    # Execute the tool
    observation = tool_function(**arguments)

    # Record the execution
    step = {
        "step": len(steps) + 1,
        "tool": tool_name,
        "arguments": arguments,
        "observation": observation
    }

    steps.append(step)

    # Add tool result to conversation history
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": tool_name,
        "content": json.dumps(observation)
    })

    return observation

In [159]:
print(execute_tool_call)

<function execute_tool_call at 0x000001FCAAD8AEF0>


## 8.4 Update checkpoint saving

### From the 8.1 def save_checkpoint funtion just update the llm_calls.

In [160]:
def save_checkpoint(
    run_id,
    scenario_id,
    messages,
    tokens_used,
    steps,
    llm_calls
):
    checkpoint = {
        "run_id": run_id,
        "scenario_id": scenario_id,
        "messages": messages,
        "tokens_used": tokens_used,
        "llm_calls": llm_calls,
        "step_number": len(steps),
        "steps": steps,
        "saved_at": datetime.now(timezone.utc).isoformat()
    }

    checkpoint_path = CHECKPOINT_DIR / f"{run_id}.json"

    with checkpoint_path.open("w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=4)

    return checkpoint_path

In [161]:
print(save_checkpoint)  #<function save_checkpoint at 0x000002BBFFA7E830> - This is for previous save_checkpoint function address.

<function save_checkpoint at 0x000001FCAAD8AD40>


## 8.5 Create the resumable agent function
    Now we'll build the main function that can execute the agent and save a checkpoint after every tool.

In [162]:
def run_agent_resumable(
    user_message,
    scenario_id,
    run_id,
    crash_after=None
):

    # Initialize conversation
    messages = [
        {
            "role": "system",
            "content": """
            AGENT
            You are an Instamart refund processing agent.
            Follow the refund policy and use available tools.
            Do not invent order information or refund decisions.
            """
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Initialize tracking variables
    steps = []
    tokens_used = 0
    llm_calls = 0

    # Agent execution loop
    for iteration in range(6):

        # Call the mock LLM
        response = chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        assistant_message = response.choices[0].message

        # Check whether the model returned a final answer
        if not assistant_message.tool_calls:
            return {
                "scenario_id": scenario_id,
                "status": assistant_message.content.split(":", 1)[0].strip(),
                "final_answer": assistant_message.content,
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        # Save assistant tool-call message
        tool_calls_data = []

        for tool_call in assistant_message.tool_calls:
            tool_calls_data.append({
                "id": tool_call.id,
                "name": tool_call.name,
                "arguments": tool_call.arguments
            })

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": tool_calls_data
        })

        # Execute requested tools
        for tool_call in assistant_message.tool_calls:

            execute_tool_call(tool_call, messages, steps)

            # Save checkpoint after each tool
            save_checkpoint(
                run_id,
                scenario_id,
                messages,
                tokens_used,
                steps,
                llm_calls
            )

            print("Checkpoint saved after:", tool_call.name)

            # Simulate crash after requested tool count
            if crash_after is not None and len(steps) == crash_after:

                raise RuntimeError(
                    "Simulated crash after tool execution"
                )

    return {
        "scenario_id": scenario_id,
        "status": "FALLBACK_MAX_ITERATIONS",
        "final_answer": "FALLBACK_MAX_ITERATIONS: Request requires human review.",
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used
    }

In [163]:
print(run_agent_resumable)

<function run_agent_resumable at 0x000001FCFA520DC0>


## 8.6 Simulate a crash after the second tool

In [164]:
# Select scenario S1
s1_row = agent_scenarios[
    agent_scenarios["scenario_id"] == "S1"
].iloc[0]

# Create a unique run ID
run_id = "S1_CRASH_TEST"

# Start the agent and simulate a crash after 2 tools
try:
    crash_result = run_agent_resumable(
        user_message=s1_row["message"],
        scenario_id="S1",
        run_id=run_id,
        crash_after=2
    )

except RuntimeError as e:
    print("Agent interrupted:", e)

Checkpoint saved after: get_order
Checkpoint saved after: search_policy
Agent interrupted: Simulated crash after tool execution


## 8.7 Inspect the saved checkpoint

In [165]:
# Load the saved checkpoint
checkpoint = load_checkpoint(run_id)

# Display checkpoint details
print("Run ID:", checkpoint["run_id"])
print("Scenario ID:", checkpoint["scenario_id"])
print("Step number:", checkpoint["step_number"])
print("LLM calls:", checkpoint["llm_calls"])
print("Tokens used:", checkpoint["tokens_used"])

print("\nTools completed before crash:")

for step in checkpoint["steps"]:
    print(step["step"], "-", step["tool"])

Run ID: S1_CRASH_TEST
Scenario ID: S1
Step number: 2
LLM calls: 2
Tokens used: 140

Tools completed before crash:
1 - get_order
2 - search_policy


## 8.8 Create the resume function

In [166]:
def resume_agent(run_id):

    # Load the saved checkpoint
    checkpoint = load_checkpoint(run_id)

    if "error" in checkpoint:
        return checkpoint

    # Restore previous execution state
    messages = checkpoint["messages"]  # The function retrieves the previous conversation and completed tool history.
    steps = checkpoint["steps"]
    tokens_used = checkpoint["tokens_used"]
    llm_calls = checkpoint["llm_calls"]
    scenario_id = checkpoint["scenario_id"]

    print("Resuming run:", run_id)
    print("Previously completed tools:", len(steps))

    # Continue the agent loop
    for iteration in range(6):

        # Call the mock LLM using saved conversation history
        response = chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        assistant_message = response.choices[0].message

        # Check for final answer
        if not assistant_message.tool_calls:

            final_answer = assistant_message.content

            return {
                "scenario_id": scenario_id,
                "status": final_answer.split(":", 1)[0].strip(),
                "final_answer": final_answer,
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        # Store assistant tool-call message
        tool_calls_data = []

        for tool_call in assistant_message.tool_calls:
            tool_calls_data.append({
                "id": tool_call.id,
                "name": tool_call.name,
                "arguments": tool_call.arguments
            })

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": tool_calls_data
        })

        # Execute remaining tools
        for tool_call in assistant_message.tool_calls:

            execute_tool_call(tool_call, messages, steps)

            # Save updated checkpoint
            save_checkpoint(
                run_id,
                scenario_id,
                messages,
                tokens_used,
                steps,
                llm_calls
            )

            print("Resumed tool executed:", tool_call.name)

    return {
        "scenario_id": scenario_id,
        "status": "FALLBACK_MAX_ITERATIONS",
        "final_answer": "FALLBACK_MAX_ITERATIONS: Request requires human review.",
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used
    }

In [167]:
print(resume_agent)

<function resume_agent at 0x000001FCAAD8BB50>


## 8.9 Resume the agent after the crash

In [168]:
# Resume the interrupted S1 agent
resumed_result = resume_agent(run_id)

print("\nFinal status:", resumed_result["status"])
print("Final answer:", resumed_result["final_answer"])
print("Total LLM calls:", resumed_result["llm_calls"])
print("Total tokens used:", resumed_result["tokens_used"])

print("\nComplete tool execution history:")

display(pd.DataFrame(resumed_result["steps"]))

Resuming run: S1_CRASH_TEST
Previously completed tools: 2
Resumed tool executed: calculate_refund
Resumed tool executed: issue_refund

Final status: REFUND
Final answer: REFUND: INR 128.25 issued.
Total LLM calls: 5
Total tokens used: 1016

Complete tool execution history:


,step,tool,arguments,observation
0,1,get_order,{'order_id': 'ORD-100048'},"{'order_id': 'ORD-100048', 'status': 'complete..."
1,2,search_policy,{'query': 'refund policy late delivery'},"{'results': [{'source': 'refund_policy.txt', '..."
2,3,calculate_refund,"{'order_amount': 128.25, 'delivery_minutes': 6...","{'refund_inr': 128.25, 'decision': 'REFUND', '..."
3,4,issue_refund,"{'order_id': 'ORD-100048', 'refund_inr': 128.25}",{'refund_inr': 128.25}


## 8.10 Verification step

In [169]:
print("Final status:", resumed_result["status"])

print("\nTools executed in order:")

for step in resumed_result["steps"]:
    print(step["step"], "-", step["tool"])

print("\nTotal tools executed:", len(resumed_result["steps"]))

print(
    "\nCheckpoint step number:",
    load_checkpoint(run_id)["step_number"]
)

Final status: REFUND

Tools executed in order:
1 - get_order
2 - search_policy
3 - calculate_refund
4 - issue_refund

Total tools executed: 4

Checkpoint step number: 4


# 9. ReAct versus plan-and-execute 

## 9.1 What we done in this task.

              ReAct	                            Plan-and-Execute
             
    1. Thinks and acts step by step	         Creates a plan before executing
    
    2. Reacts to tool results	               Follows the planned sequence
    
    3. Adapts to unexpected observations	   May need replanning

In [176]:
# Create a summary of the original agent results

comparison_data = []

for result in agent_results:

    comparison_data.append({
        "Scenario": result["scenario_id"],
        "Status": result["status"],
        "LLM Calls": result["llm_calls"],
        "Tokens Used": result["tokens_used"],
        "Tools Executed": len(result["steps"])
    })

comparison_df = pd.DataFrame(comparison_data)

display(comparison_df)

,Scenario,Status,LLM Calls,Tokens Used,Tools Executed
0,S1,REFUND,5,1021,4
1,S2,REFUND,5,1021,4
2,S3,ESCALATE,5,1004,4
3,S4,DENIED,4,697,3
4,S5,DENIED,3,397,2
5,S6,NOT_FOUND,2,112,1
6,S7,FALLBACK_MAX_ITERATIONS,6,3120,6


In [177]:
from pathlib import Path
import json

# Define output files
agent_runs_path = OUTPUT_DIR / "agent_runs.jsonl"
ledger_path = OUTPUT_DIR / "refund_ledger.jsonl"

print("AGENT RUNS FILE EXISTS:", agent_runs_path.exists())
print("REFUND LEDGER EXISTS:", ledger_path.exists())
print("CHECKPOINT FOLDER EXISTS:", CHECKPOINT_DIR.exists())

# Count agent runs
if agent_runs_path.exists():
    with agent_runs_path.open("r", encoding="utf-8") as f:
        agent_lines = [json.loads(line) for line in f if line.strip()]

    print("\nTotal agent runs:", len(agent_lines))

    required_fields = {
        "scenario_id",
        "status",
        "final_answer",
        "llm_calls",
        "tokens_used",
        "steps"
    }

    for i, run in enumerate(agent_lines):
        missing = required_fields - set(run.keys())

        if missing:
            print(f"Run {i}: Missing fields:", missing)

    print("Required fields checked.")

# Count ledger entries
if ledger_path.exists():
    with ledger_path.open("r", encoding="utf-8") as f:
        ledger_lines = [json.loads(line) for line in f if line.strip()]

    print("\nTotal refund ledger entries:", len(ledger_lines))

    for entry in ledger_lines:
        print(entry)

AGENT RUNS FILE EXISTS: True
REFUND LEDGER EXISTS: True
CHECKPOINT FOLDER EXISTS: True

Total agent runs: 7
Required fields checked.

Total refund ledger entries: 4
{'timestamp': '2026-11-08T06:54:52.716205+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-08T06:54:52.721447+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100067', 'refund_inr': 135.0}
{'timestamp': '2026-11-08T06:54:52.727543+00:00', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0}
{'timestamp': '2026-11-08T06:54:53.287658+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}


In [178]:
print("Project directory:", BASE_DIR)

readme_path = BASE_DIR / "README.md"

print("README exists:", readme_path.exists())

if readme_path.exists():
    print(readme_path.read_text(encoding="utf-8"))

Project directory: c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code
README exists: True
# Instamart Copilot Project

This repository contains the completed Instamart assignment (Parts A-D), its original Jupyter notebooks, datasets, SQLite database, mock LLM, SQL queries, and generated outputs.

## Project Overview

This project covers data cleaning, customer analytics, machine learning, support-ticket triage, and a policy-based refund assistant.

## Project structure

```text
Instamart_GitHub_Ready/
├── data/                 # Original and cleaned datasets, policies, SQLite database
├── notebooks/            # Data_generate_code.ipnby ,mock_llm.ipynb & .py (preserved)
├── src/
│   ├── part_a.py          # Part A notebook runner
│   ├── part_b.py          # Part B notebook runner
│   ├── part_c.py          # Part C notebook runner
│   ├── part_d.py          # Part D notebook runner
│   ├── notebook_runner.py # Shared notebook execution helper
│   └── main.py          

# ---------- PART D is Done ----------